# 05b. Bandit Offline con Replay Simplificado

Este notebook deja un experimento comparativo de bandit contextual ligero usando un flujo cronológico de interacciones históricas.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

import numpy as np

from src.utils.recsys_utils import ensure_dir, sample_recent_interactions

processed_dir = ROOT / 'data' / 'processed'
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')

In [2]:
train = pd.read_parquet(processed_dir / 'interactions_train.parquet')
valid = pd.read_parquet(processed_dir / 'interactions_valid.parquet')
test = pd.read_parquet(processed_dir / 'interactions_test.parquet')

## Preparar flujo cronológico y recompensas

Para mantener tiempos razonables, se usa una ventana reciente y un conjunto reducido de ítems frecuentes.


In [3]:
stream = pd.concat([train, valid, test], ignore_index=True).sort_values('timestamp')
stream = sample_recent_interactions(stream, n_rows=150_000)
frequent_items = stream.groupby('movieId').size().loc[lambda s: s >= 80].index
stream = stream[stream['movieId'].isin(frequent_items)].copy()
stream['reward'] = (stream['rating'] >= 4.0).astype(int)

item_reward_mean = stream.groupby('movieId')['reward'].mean().sort_values(ascending=False)
oracle_item = int(item_reward_mean.index[0])
oracle_reward = float(item_reward_mean.iloc[0])
print('Eventos usados:', len(stream))
print('Item oracle:', oracle_item, 'reward medio:', round(oracle_reward, 4))


Eventos usados: 23467
Item oracle: 166024 reward medio: 0.8723


## Simular políticas

Se compara una política greedy por media empírica contra `epsilon-greedy` con replay simplificado.


In [4]:
results = []
for policy_name, epsilon in [('greedy', 0.0), ('epsilon_greedy', 0.1)]:
    item_stats = {}
    cumulative_reward = 0
    matched_events = 0

    for row in stream.itertuples(index=False):
        available_items = list(item_reward_mean.index[:100])
        explore = np.random.rand() < epsilon or not item_stats
        if explore:
            chosen_item = int(np.random.choice(available_items))
        else:
            chosen_item = max(item_stats, key=lambda item: item_stats[item]['mean'])

        if chosen_item == int(row.movieId):
            matched_events += 1
            reward = int(row.reward)
            cumulative_reward += reward
            stats = item_stats.setdefault(chosen_item, {'sum': 0, 'count': 0, 'mean': 0.0})
            stats['sum'] += reward
            stats['count'] += 1
            stats['mean'] = stats['sum'] / stats['count']

    avg_reward = cumulative_reward / matched_events if matched_events else 0.0
    regret = max(0.0, oracle_reward - avg_reward)
    results.append({
        'policy': policy_name,
        'matched_events': matched_events,
        'cumulative_reward': cumulative_reward,
        'average_reward': avg_reward,
        'estimated_regret': regret,
    })

bandit_metrics = pd.DataFrame(results)
display(bandit_metrics)
bandit_metrics.to_csv(tables_dir / 'bandit_metrics.csv', index=False)


,policy,matched_events,cumulative_reward,average_reward,estimated_regret
0,greedy,89,55,0.617978,0.254363
1,epsilon_greedy,122,78,0.639344,0.232996
